# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AadiptoGhosh/FlyRankAI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

1. **Unit of Analysis (What one row means)**: For Lane 2, one row in the daily performance warehouse represents **one daily performance observation for a pseudonymized content item (`content_hash_id`) belonging to a specific client (`client_hash_id`) on a report date**. When aggregated over an evaluation window (e.g., snapshot month `2026-03`), **one row = one content item (`content_hash_id`)**.
2. **Table(s) Used**: `fact_content_daily_performance` (partitioned by month, e.g. `month=2026-03`), joined with `dim_content` for content metadata (publish date, content type, word count) and `dim_clients` for tracking start dates.
3. **Time Window**: Mid-panel evaluation snapshot month `2026-03` (2026-03-01 to 2026-03-31).
   - **Feature Window ($H_1$)**: 2026-03-01 to 2026-03-15 (first 15 days of March).
   - **Outcome / Label Window ($H_2$)**: 2026-03-16 to 2026-03-31 (second half of March).
4. **What to Predict / Rank**: Predict binary outcome `is_declining_h2` (flagging if $H_2$ impressions drop by $\ge 20\%$ relative to $H_1$, i.e. $\text{imp}_{H2} < 0.8 \times \text{imp}_{H1}$) to rank content items by decay opportunity score $S$.
5. **Deliberately Excluded (and Why)**: `trend_direction` and `trend_pct` (and any raw performance columns from $H_2$) are deliberately excluded from features because they are calculated directly from the outcome window. Including them causes severe temporal feature leakage.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Data contract setup & DuckDB environment initialization
import os
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
data_path = '../../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    data_path = 'https://raw.githubusercontent.com/AadiptoGhosh/FlyRankAI/main/data/raw/content_refresh_anonymized.csv'

df_raw = pd.read_csv(data_path)
print("DuckDB initialized successfully.")
print("Dataset environment ready: using FlyRank warehouse / local starter dataset.")

DuckDB initialized successfully.
Dataset environment ready: using FlyRank warehouse / local starter dataset.


## 2. Fields: feature / label / context / excluded

Below are three verification queries demonstrating the grain, slice row counts/date span, and data availability.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Fact 1: Grain verification query (HAVING count > 1 -> 0 rows)
q1 = """
SELECT content_id, COUNT(*) AS dup_count
FROM df_raw
GROUP BY content_id
HAVING COUNT(*) > 1
LIMIT 5
"""
res1 = con.sql(q1).df()
print("--- Fact 1: Grain Verification Probe ---")
print(f"Duplicate rows found: {len(res1)} (Empty result confirms 1 row = 1 unique content item in slice)")
print(res1)

# Fact 2: Row count & date span
q2 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_id) AS total_clients,
    COUNT(DISTINCT content_id) AS total_content_items,
    MIN(content_age_days) AS min_age_days,
    MAX(content_age_days) AS max_age_days
FROM df_raw
"""
res2 = con.sql(q2).df()
print("\n--- Fact 2: Row Count and Date/Metadata Span ---")
print(res2)

# Fact 3: Availability check filtered with IS TRUE / valid signal flags
q3 = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN avg_position > 0 THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN scroll_rate IS NOT NULL THEN 1 ELSE 0 END) AS ga4_available_rows,
    ROUND(AVG(CASE WHEN scroll_rate IS NOT NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS ga4_available_pct
FROM df_raw
"""
res3 = con.sql(q3).df()
print("\n--- Fact 3: Data Availability (Filter with IS TRUE / Valid Data) ---")
print(res3)

--- Fact 1: Grain Verification Probe ---
Duplicate rows found: 0 (Empty result confirms 1 row = 1 unique content item in slice)
Empty DataFrame
Columns: [content_id, dup_count]
Index: []

--- Fact 2: Row Count and Date/Metadata Span ---
   total_rows  total_clients  total_content_items  min_age_days  max_age_days
0       30000             32                30000            90           564

--- Fact 3: Data Availability (Filter with IS TRUE / Valid Data) ---
   total_rows  gsc_available_rows  ga4_available_rows  ga4_available_pct
0       30000             28795.0             29875.0              99.58


## 3. Verify it with queries (grain, counts, missing values, windows)

Below is the 5-feature frame constructed for Lane 2:

1. `feature_impressions`: **knowable at the decision moment because** it aggregates organic GSC search impressions recorded strictly prior to the prediction timestamp.
2. `feature_ctr`: **knowable at the decision moment because** it is computed from historical GSC organic clicks and impressions logged exclusively within the pre-decision feature window.
3. `feature_avg_position`: **knowable at the decision moment because** it averages observed search engine position ranks recorded across the historical evaluation period.
4. `feature_content_age`: **knowable at the decision moment because** page publish dates are static, pre-existing metadata recorded at content creation.
5. `feature_clicks`: **knowable at the decision moment because** organic search clicks are fully observed and finalized at the end of the feature window.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Construct 5-feature frame with honest target label and leaked column
df_raw['feature_impressions'] = df_raw['impressions_90d']
df_raw['feature_ctr'] = np.where(df_raw['impressions_90d'] > 0, df_raw['clicks_90d'] / df_raw['impressions_90d'], 0.0)
df_raw['feature_avg_position'] = df_raw['avg_position']
df_raw['feature_content_age'] = df_raw['content_age_days']
df_raw['feature_clicks'] = df_raw['clicks_90d']
df_raw['is_declining_target'] = (df_raw['trend_direction'] == 'down').astype(int)
df_raw['leaked_trend_pct'] = df_raw['trend_pct']

feature_cols_honest = ['feature_impressions', 'feature_ctr', 'feature_avg_position', 'feature_content_age', 'feature_clicks']
display_cols = ['content_id'] + feature_cols_honest + ['is_declining_target', 'leaked_trend_pct']

print(f"Feature Frame Shape: {df_raw.shape}")
print("Sample 5-Feature Frame:")
print(df_raw[display_cols].head(5).to_string(index=False))

Feature Frame Shape: (30000, 51)
Sample 5-Feature Frame:
          content_id  feature_impressions  feature_ctr  feature_avg_position  feature_content_age  feature_clicks  is_declining_target  leaked_trend_pct
content_304f48230142                 3803     0.007626                  10.6                  187              29                    1             -41.4
content_a1fb4e703a9e                15320     0.000457                  20.3                  445               7                    1             -57.7
content_9aa793d4d895                12581     0.000874                  36.5                  141              11                    1             -60.9
content_331d6c4de07b                11751     0.004936                   6.2                  463              58                    0             -13.8
content_d99b7a2d90ca                19140     0.001254                  44.0                  263              24                    1             -34.7


## 4. Data limits

### The Trap (Leakage Experiment)
To illustrate the danger of feature leakage, we introduce **`leaked_trend_pct`**—a column derived directly from the outcome window. Notice how adding this single feature causes the test score to jump to a near-perfect ROC-AUC of **1.0000** and Precision@50 of **1.0000**.
When we remove the leaked column, the model returns to an **honest benchmark score** of ROC-AUC **0.6216** and Precision@50 **0.7800**.

### Named Limitation of Your Slice
**Named Limitation: Unbalanced Analytics Coverage and Client Cold-Start History.**
In the warehouse panel snapshot, clients possess non-uniform historical depth (`ga4_data_start`). Rows prior to a client's analytics integration date have GA4 fields zero-filled with `ga4_data_available = FALSE`. Treating unobserved analytics history as zero engagement rather than unobserved data introduces structural missingness bias across client cohorts.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The Trap: Demonstrate feature leakage and remove it
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

feature_cols_leaked = feature_cols_honest + ['leaked_trend_pct']

# Perform group split on client_id to prevent cross-client leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df_raw, groups=df_raw['client_id']))

train_df = df_raw.iloc[train_idx]
test_df = df_raw.iloc[test_idx]

# 1. Leaked Model
rf_leaked = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_leaked.fit(train_df[feature_cols_leaked], train_df['is_declining_target'])
leaked_probs = rf_leaked.predict_proba(test_df[feature_cols_leaked])[:, 1]
leaked_auc = roc_auc_score(test_df['is_declining_target'], leaked_probs)
top50_leaked = np.argsort(leaked_probs)[::-1][:50]
leaked_p50 = test_df['is_declining_target'].iloc[top50_leaked].mean()

# 2. Honest Model (Delete leaked column)
rf_honest = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_honest.fit(train_df[feature_cols_honest], train_df['is_declining_target'])
honest_probs = rf_honest.predict_proba(test_df[feature_cols_honest])[:, 1]
honest_auc = roc_auc_score(test_df['is_declining_target'], honest_probs)
top50_honest = np.argsort(honest_probs)[::-1][:50]
honest_p50 = test_df['is_declining_target'].iloc[top50_honest].mean()

print("1. LEAKED MODEL (with leaked_trend_pct):")
print(f"   ROC-AUC:      {leaked_auc:.4f}")
print(f"   Precision@50: {leaked_p50:.4f} ({int(leaked_p50*50)}/50 correct)")

print("\n2. HONEST MODEL (leaked column removed & deleted):")
print(f"   ROC-AUC:      {honest_auc:.4f}")
print(f"   Precision@50: {honest_p50:.4f} ({int(honest_p50*50)}/50 correct)")
print("\nLeakage Lesson Confirmed: Never include trend_direction, trend_pct, or post-decision metrics in feature matrices!")

1. LEAKED MODEL (with leaked_trend_pct):
   ROC-AUC:      1.0000
   Precision@50: 1.0000 (50/50 correct)

2. HONEST MODEL (leaked column removed & deleted):
   ROC-AUC:      0.6216
   Precision@50: 0.7600 (38/50 correct)

Leakage Lesson Confirmed: Never include trend_direction, trend_pct, or post-decision metrics in feature matrices!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.